# MicroPython: Python auf Mikrocontrollern

Vielleicht sind Ihnen schon kleine Platinen wie der *Raspberry Pi Pico* oder ein *ESP32* begegnet, etwa in einem Laborpraktikum oder einem Bastelprojekt. Auf solchen **Mikrocontrollern** läuft kein Betriebssystem wie Windows oder Linux, und der Arbeitsspeicher ist mit einigen Hundert Kilobyte winzig. Trotzdem lassen sie sich in Python programmieren – mit **MicroPython**, einer schlanken Umsetzung von Python 3 für Mikrocontroller.

Dieser Anhang ist kein Kurs, sondern soll Ihnen einen ersten Eindruck vermitteln: Was ist gleich, was ist anders, und wie sieht ein erstes Programm aus? Als Beispiel dient der Raspberry Pi Pico. Hardware brauchen Sie dafür nicht: Die Beispiele lassen sich im Simulator [Wokwi](https://wokwi.com) direkt im Browser ausprobieren.

## Was ist gleich, was ist anders?

Fast alles, was Sie in diesem Skript gelernt haben, gilt auch in MicroPython: Variablen, Datentypen, Verzweigungen, Schleifen, Funktionen, Listen und Dictionaries, sogar Klassen. Die Unterschiede liegen vor allem in der Umgebung:

| | Python auf dem PC | MicroPython auf dem Mikrocontroller |
|---|---|---|
| Speicher | mehrere Gigabyte | einige Hundert Kilobyte |
| Bibliotheken | riesige Auswahl, z.B. NumPy, pandas, Matplotlib | kleine Standardbibliothek; NumPy, pandas und Matplotlib gibt es nicht |
| Hardwarezugriff | nur über Zusatzpakete | eingebaut: Modul `machine` für Pins, Analog-Digital-Umsetzer usw. |
| Programmstart | Skript oder Notebook-Zelle ausführen | Datei `main.py` startet automatisch nach dem Einschalten |
| Programmende | nach der letzten Anweisung | oft nie: Endlosschleife `while True:` |

Der letzte Punkt ist typisch für eingebettete Systeme: Ein Mikrocontroller, der eine Temperatur überwacht, soll das ohne Unterbrechung tun, solange er mit Strom versorgt wird. Die Endlosschleife, die im Kapitel *Kontrollstrukturen* als gefürchteter Fehler auftauchte, ist hier also ausdrücklich gewollt.

## Das „Hello, World“ der Mikrocontroller: eine blinkende LED

Ein Mikrocontroller hat oft kein Display, auf dem man „Hello, World“ ausgeben könnte. Das klassische erste Programm lässt deshalb eine LED blinken. Der Raspberry Pi Pico hat dafür eine LED direkt auf der Platine:

In [ ]:
#| eval: false
from machine import Pin
import time

led = Pin("LED", Pin.OUT)  # eingebaute LED als Ausgang

while True:
    led.toggle()           # LED umschalten: an -> aus -> an ...
    time.sleep(0.5)        # 0,5 Sekunden warten

Einige Details:

- Ein **Pin** ist ein Anschluss des Mikrocontrollers. `Pin.OUT` macht ihn zu einem Ausgang, der entweder eine Spannung ausgibt (an) oder nicht (aus). `"LED"` ist ein vordefinierter Name für den Pin, an dem die eingebaute LED hängt. Externe Bauteile sprechen Sie über die Nummer des Pins an, z.B. `Pin(15, Pin.OUT)`.
- Statt `toggle()` können Sie die LED auch gezielt mit `led.on()` und `led.off()` schalten.
- Das Modul `time` kennen Sie vom PC. Gestoppt wird das Programm in der Entwicklungsumgebung mit dem Stopp-Button oder mit Strg+C.

## Eine Spannung messen

Viele Sensoren, z.B. für Temperatur, Druck oder Helligkeit, liefern eine **analoge Spannung**. Damit kann ein Mikrocontroller nicht direkt rechnen: Ein **Analog-Digital-Umsetzer** (*Analog-to-Digital Converter*, ADC) wandelt die Spannung zunächst in eine ganze Zahl um. In MicroPython liefert die Methode `read_u16()` einen Wert zwischen 0 und 65535 (also $2^{16}-1$, eine 16-Bit-Zahl; der ADC des Pico misst intern mit 12 Bit, MicroPython rechnet den Wert auf diesen einheitlichen Bereich um). Beim Pico entspricht das dem Spannungsbereich von 0 bis 3,3 V.

Das folgende Programm misst einmal pro Sekunde die Spannung an Pin GP26, an dem z.B. ein Potentiometer oder ein Sensor angeschlossen ist, rechnet sie in Volt um und schaltet die LED ein, solange ein Grenzwert überschritten ist:

In [ ]:
#| eval: false
from machine import Pin, ADC
import time

adc = ADC(Pin(26))         # Analog-Digital-Umsetzer an Pin GP26
led = Pin("LED", Pin.OUT)
limit_V = 2.5              # Grenzwert in Volt

while True:
    raw = adc.read_u16()           # Rohwert zwischen 0 und 65535
    voltage = raw * 3.3 / 65535    # Umrechnung in Volt
    print(f"Rohwert: {raw:5d}   Spannung: {voltage:.2f} V")
    if voltage > limit_V:
        led.on()
    else:
        led.off()
    time.sleep(1)

Bis auf den Zugriff auf die Hardware über das Modul `machine` enthält das Programm nur Bekanntes: eine Schleife, eine Verzweigung, eine Umrechnung mit Gleitkommazahlen und eine formatierte Ausgabe. Die `print()`-Ausgaben erscheinen übrigens nicht auf dem Mikrocontroller selbst, sondern werden über das USB-Kabel an die Konsole der Entwicklungsumgebung übertragen.

::: {.callout-note}
### Andere Boards {.unnumbered}

Auf anderen Mikrocontrollern, z.B. einem ESP32, funktioniert der Code fast genauso. Unterschiede gibt es vor allem bei den Pin-Nummern (nicht jedes Board hat eine eingebaute LED mit dem Namen `"LED"`) und beim Messbereich des ADC. Die Kurzreferenz (*Quick reference*) für das jeweilige Board in der MicroPython-Dokumentation nennt diese Details.
:::

## REPL und `main.py`

Eine einsteigerfreundliche Entwicklungsumgebung für MicroPython ist [Thonny](https://thonny.org). Zunächst wird einmalig die MicroPython-Firmware auf den Mikrocontroller kopiert; Thonny bietet dafür eine eigene Installationsfunktion. Danach verbindet sich Thonny über USB mit dem Board, und es gibt zwei Arbeitsweisen:

- **REPL** (*Read-Eval-Print Loop*): eine interaktive Konsole, die direkt auf dem Mikrocontroller läuft. Jede eingegebene Zeile wird sofort ausgeführt – ideal, um z.B. nach `led = Pin("LED", Pin.OUT)` die LED testweise mit `led.on()` einzuschalten. Das entspricht dem schrittweisen Ausprobieren in einer Notebook-Zelle.
- **Datei `main.py`:** Ein Programm, das unter diesem Namen auf dem Mikrocontroller gespeichert ist, startet nach dem Einschalten automatisch – auch ohne angeschlossenen PC, z.B. mit einer Batterie als Stromversorgung.

::: {.callout-tip}
### Ausprobieren ohne Hardware {.unnumbered}

Wählen Sie auf [wokwi.com](https://wokwi.com) den Raspberry Pi Pico mit MicroPython als Vorlage, fügen Sie das Blink-Programm in die Datei `main.py` ein und starten Sie die Simulation. Für das Messprogramm ergänzen Sie ein Potentiometer und verbinden dessen mittleren Anschluss mit GP26 und die beiden äußeren mit 3V3 und GND. Drehen Sie während der Simulation am Potentiometer und beobachten Sie die Ausgabe und die LED.
:::

## Zum Weiterlesen

- [MicroPython-Dokumentation](https://docs.micropython.org) mit Kurzreferenzen für die einzelnen Boards, z.B. für den [Raspberry Pi Pico](https://docs.micropython.org/en/latest/rp2/quickref.html)
- [Wokwi](https://wokwi.com): Simulator für Mikrocontroller im Browser, mit LEDs, Potentiometern und vielen Sensoren
- [Thonny](https://thonny.org): einfache Python-Entwicklungsumgebung mit MicroPython-Unterstützung